# MotoTrack - fine-tune a mixed-traffic detector (optional)

This notebook is **optional**. MotoTrack works with pretrained weights and no training at all:
`scripts/fetch_models.py` pulls `yolo26n.pt` and the Visaitech mixed-traffic checkpoint, and the
pipeline runs immediately.

Train here only if you want to see whether fine-tuning beats the pretrained baseline **on your own
gold set**. That comparison is the point, not the training itself.

## What this notebook does

1. Checks for a GPU (Colab free tier gives a T4).
2. Downloads the Visaitech mixed-traffic dataset, **skipping the 12 GB of raw videos** (only the
   labelled frames are needed).
3. Rewrites `data.yaml` with an absolute path. *This matters:* the published `data.yaml` uses
   `path: .`, which Ultralytics resolves against the current working directory, not the dataset
   folder. Left as-is, training fails with `images not found`.
4. Fine-tunes `yolo26n.pt` from the official COCO checkpoint.
5. Saves every epoch to Drive, so a Colab disconnect does not lose the run.
6. Evaluates and exports `best.pt` for use in MotoTrack.

## Before you start

**Runtime > Change runtime type > T4 GPU.** CPU training works but is far slower.

Expect roughly 1-2 hours for 100 epochs on a T4. The dataset is small (293 frames), so this is
not a heavy job.

## 1. Check the runtime

In [ ]:
import subprocess, sys, os

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode == 0:
    print('GPU:', gpu.stdout.strip())
else:
    print('No GPU detected. Runtime > Change runtime type > T4 GPU.')
    print('Training will still run on CPU, just slowly.')

print('Python:', sys.version.split()[0])

## 2. Install dependencies

In [ ]:
!pip install -q 'ultralytics>=8.4.63' huggingface_hub

import ultralytics, torch
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

## 3. Mount Drive for checkpoint safety

Colab free tier disconnects. Every epoch is written to Drive so a disconnect costs at most one
epoch, not the whole run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/mototrack'
os.makedirs(PROJECT, exist_ok=True)
print('project dir:', PROJECT)

## 4. Download the labelled frames only

The repo also carries 203 raw clips (~12 GB) that are not needed for training. `ignore_patterns`
skips them, so this is a small download.

In [ ]:
from huggingface_hub import snapshot_download

REPO = 'visaitech/vehicle-mixed-traffic-detection'
DATA_DIR = snapshot_download(
    repo_id=REPO,
    repo_type='dataset',
    allow_patterns=['data.yaml', 'classes.txt', 'images/train/*', 'images/val/*', 'labels/train/*', 'labels/val/*'],
    ignore_patterns=['videos/*'],
    local_dir='/content/dataset',
)
print('dataset at:', DATA_DIR)

for split in ('train', 'val'):
    imgs = len(os.listdir(os.path.join(DATA_DIR, 'images', split)))
    lbls = len(os.listdir(os.path.join(DATA_DIR, 'labels', split)))
    print(f'{split}: {imgs} images, {lbls} label files')

## 5. Fix `data.yaml` (do not skip)

The published file says `path: .`. Ultralytics resolves that against the **current working
directory**, so it looks for `/content/images/val` and fails with:

```
FileNotFoundError: Dataset '.../data.yaml' images not found, missing path '/content/images/val'
```

This cell rewrites `path` to the absolute dataset directory. Verified against the real file.

In [ ]:
import yaml

with open(os.path.join(DATA_DIR, 'data.yaml')) as fh:
    cfg = yaml.safe_load(fh)

cfg['path'] = DATA_DIR
cfg.setdefault('train', 'images/train')
cfg.setdefault('val', 'images/val')

fixed = os.path.join(DATA_DIR, 'mototrack_data.yaml')
with open(fixed, 'w') as fh:
    yaml.safe_dump(cfg, fh, sort_keys=False)

print(open(fixed).read())

## 6. Verify the dataset resolves before spending an hour on training

In [ ]:
from ultralytics.data.utils import check_det_dataset

resolved = check_det_dataset(fixed)
print('names :', resolved['names'])
print('train :', resolved['train'], '| exists:', os.path.exists(resolved['train']))
print('val   :', resolved['val'], '| exists:', os.path.exists(resolved['val']))

assert os.path.exists(resolved['train']), 'train split did not resolve'
assert os.path.exists(resolved['val']), 'val split did not resolve'
print('\nOK - dataset is ready')

## 7. Fine-tune

Started from `yolo26n.pt` (COCO), not from scratch: 293 labelled frames is far too little to
train a detector from zero, and fine-tuning is what the reference implementation did too.

`project=PROJECT` writes every epoch to Drive.

In [ ]:
from ultralytics import YOLO

EPOCHS = 100
IMGSZ = 640
BATCH = 16   # lower to 8 if you hit out-of-memory on a T4

model = YOLO('yolo26n.pt')
results = model.train(
    data=fixed,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name='mixed_finetune',
    exist_ok=True,
    patience=30,
    seed=0,
    plots=True,
)

## 8. Validate and read the real numbers

In [ ]:
BEST = os.path.join(PROJECT, 'mixed_finetune', 'weights', 'best.pt')
print('best checkpoint:', BEST, '| exists:', os.path.exists(BEST))

tuned = YOLO(BEST)
metrics = tuned.val(data=fixed, imgsz=IMGSZ)

print(f"mAP50    : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map:.3f}")
print(f"precision: {metrics.box.mp:.3f}")
print(f"recall   : {metrics.box.mr:.3f}")
print('\nCompare these to the published baseline in the model card:')
print('  mAP50 0.719 / recall 0.734 on original frames, mAP50 0.629 on anonymised frames.')
print('  With 293 training frames, do not expect to beat those by much.')

## 9. Export `best.pt` for MotoTrack

Download it, then drop it into your local `models/` folder. `MixedTrafficDetector` expects the
filename below; overriding it is a one-line change in `config.py` if you prefer another name.

In [ ]:
import shutil
from google.colab import files

LOCAL_NAME = 'visaitech_mixed_traffic_yolo26n.pt'
exported = os.path.join('/content', LOCAL_NAME)
shutil.copy(BEST, exported)
print('exported:', exported, os.path.getsize(exported), 'bytes')

files.download(exported)
print('\nDrop this file into your local models/ directory, then run:')
print('  python -m mototrack.cli analyze clip.mp4 --detector mixed')

## 10. The part that actually matters: does it beat the baseline?

Ultralytics mAP is measured on the *dataset's* validation split, which is South Asian footage.
Your MotoTrack gold set is *your* footage. A higher mAP here does not guarantee a better verdict
accuracy there.

To answer the question that matters, run this locally with the new weights in place:

```bash
python -m mototrack.cli compare-detectors
```

and look at the binary accuracy column. If the fine-tuned model does not win on your gold set,
keep the pretrained one and write down that it did not win. A negative result measured properly is
worth more than an unmeasured positive one.